### RAG Pipeline - from Data Ingestion to Vector Database

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
### Data ingestion - To read all the pdf's in a directory

def process_all_pdf(pdf_directory):

    #process all the pdf's in the directory

    all_documents = []
    pdf_dir = Path(pdf_directory)

    #find all the pdf files

    pdf_files = list(pdf_dir.glob("**/*.pdf")) #here we are making a list of all the pdf files

    print(f"found {len(pdf_files)} pdf files to process") 

###now we're making the document out of the pdf files and also adding few extra meta data in it

    for pdf_file in pdf_files:

        print(f"\nProcessing: {pdf_file.name}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            #adding some more meta data in the documents

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"


            all_documents.extend(documents)
            print(f"loaded {len(documents)} pages")

        except Exception as e:
            print(f"error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdf("../data")





In [ ]:
## now we will chunk these documents into smaller parts to perform better in a rag application

def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):

    text_splitter = RecursiveCharacterTextSplitter(

        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", " ", ""]

    )

    split_document = text_splitter.split_documents(documents)
    print(f"SPlit {len(documents)} documents into {len(split_document)} chunks")

    if split_document:
        print(f"\nExample Chunk")
        print(f"content: {split_document[0].page_content[:200]}...")
        print(f"Metadata: {split_document[1].metadata}")

    return split_document


In [ ]:
chunks = split_documents(all_pdf_documents)
chunks

### Embedding and vector store DB

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer ##embedding model is present in here
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):

        """
        here we are initialising the embedding manager

        Args: 
            model_name = Hugging face model name for sentence embeddings

        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):

        """Load the sentence transformer model"""

        try:
            print(f"Loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Loaded Successfully. Embedding dimension: {self.model.get_embedding_dimension()}")

        except Exception as e:
            print(f"Error loading model {self.model_name}:{e}")
            raise


    def generate_embeddings(self, texts: List[str]) -> np.ndarray:

        """Generate embeddings for a list of texts
        
        Args: 
            texts: List of text strings to embed

        Returns:
            numpy array of shape (len(texts), embedding_dimension)

        """
        if not self.model:
            raise ValueError("model not loaded")

        print(f"generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generate embeddings with shape: {embeddings.shape}")
        return embeddings


###initialising the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager    


    

        

### Vector Store

In [ ]:
class VectorStore:

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vectorstore"):

        """Initiate the vector store
        
        Args: 
            collection_name = Name of the chromadb collection
            persist_directory = 

        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()


    def _initialize_store(self):
        """
        Initialize chromadb client and collection

        """

        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path = self.persist_directory)

            # get or create collection

            self.collection = self.client.get_or_create_collection(

                name = self.collection_name,
                metadata = {"description":"PDF document for RAG", "hnsw:space":"cosine"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        
        
        except Exception as e:
            print(f"Error initializing store vector: {e}")
            raise



    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        add documents and their embeddings in the vector store

        Args:
            documents: List of langchain documents
            embeddings: corresponding embeddings for the documents
        """ 
        if len(documents) != len(embeddings):
            raise ValueError("No. of documents must match the number of embeddings")

        print(f"Adding {len(documents)} documents to the vector store")

        # prepare data for chromadb

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            #generate unique id
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index']= i
            metadata['content_length']= len(doc.page_content)
            metadatas.append(metadata)

            #documents content
            documents_text.append(doc.page_content)

            #embedding
            embeddings_list.append(embedding.tolist())

        #add to collection

        try:
            self.collection.add(
                ids = ids,
                embeddings = embeddings_list,
                metadatas = metadatas,
                documents = documents_text
            )        
            print(f"successfully added {len(documents)} document to the server")
            print(f"total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"error adding documents to the vector store: {e}")
            raise


vectorstore = VectorStore()
vectorstore

In [ ]:
chunks

In [ ]:
## converting the text into the embeddings
text = [doc.page_content for doc in chunks]

## Generate the embeddings
embeddings = embedding_manager.generate_embeddings(text)

##store in the vector database
vectorstore.add_documents(chunks, embeddings)



###Retriever pipeline from vectordb

In [ ]:
class RAGRetriever:

    """Handles query based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialise the retriever

        Args: 
            vector_store: Vector store containing doc embeddings
            embedding_manager: manager for generating query embeddings
        
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager


    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant document for a query

        Args:
            query: The search query
            top_k: number of top results to return
            score_threshold: minimum similarity score threshold
        
            return:
                list of dictionaries containing retrieved documents and metadata
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f" Top K: {top_k}, and Score Threshold: {score_threshold}")

        #Generate query embeddings

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        #search in vector store

        try:

            results = self.vector_store.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results=top_k
            )

            #process results

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    #convert distances to similarity score

                    similarity_score = 1-distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'ids': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score':similarity_score,
                            'distance':distance,
                            'rank':i+1
                        })

                print(f"retrieved {len(retrieved_docs)} documents (after filtering)")

            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            raise []


rag_retriever = RAGRetriever(vectorstore, embedding_manager)
















In [ ]:
rag_retriever

## Integration of vectorDB context pipeline with LLM output 

In [ ]:
###simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

#Initialize the Groq LLM
groq_api_key = os.getenv('GROQ_API_KEY')

llm = ChatGroq(groq_api_key=groq_api_key, model_name='openai/gpt-oss-20b', temperature = 0.1, max_tokens =1024)

##2. Simple RAG function: retrieve context + generate response

def rag_simple(query, retriever, llm, top_k = 3):
    ##retrieve the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "NO relevant context found to answer the question"

    ##Generate the answer using groq llm

    prompt = f"""Use the following context to answer the question precisely

        Context:
        {context}

        Question:
        {query}

        Answer: """

    response = llm.invoke([prompt.format(context=context, query = query)])
    return response.content



In [ ]:
answer = rag_simple("what is the company's total revenue for the year?", rag_retriever, llm)
print(answer)

In [ ]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context = False):
    """RAG pipeline with some extra features.
    Returns: Answers, sources, confidence score, and full context 
        """
    results = retriever.retrieve(query, top_k=top_k, score_threshold = min_score)
    if not results:
        return{'answer':'No relevant information found', 'sources':[], 'confidence':0.0, 'context':''}

    #Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source':doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page':doc['metadata'].get('page', 'unknown'),
        'score':doc['similarity_score'],
        'preview':doc['content'][:300]+'....'
    } for doc in results]

    confidence = max([doc['similarity_score'] for doc in results])


    #Generate answers
    prompt=f"""Use the following context to answer the question properly and concisely. \nContext: \n{context}\n\nQuestion:\n{query}\n\nAnswers:"""
    repsonse=llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer':repsonse.content,
        'sources':sources,
        'confidence':confidence
    }
    if return_context:
        output['context'] = context
    return output

result = rag_advanced("What was the profit before tax", rag_retriever, llm, top_k=3, return_context=True)
print("Answer:",result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])


